In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.nn import Softmax

In [2]:
datas = pd.read_csv('基本操作/datas/text_classify/train_tokens3.csv', sep='\t', header=None, )

In [3]:
datas.head(10)

,0,1
0,还有 双鸭山 到 淮阴 的 汽车票 吗 13 号 的,Travel-Query
1,从 这里 怎么 回家,Travel-Query
2,随便 播放 一首 专辑 阁楼 里 的 佛里 的 歌,Music-Play
3,给 看 一下 墓王之王 嘛,FilmTele-Play
4,我 想 看 挑战 两把 s686 打 突变 团竞 的 游戏 视频,Video-Play
5,我 想 看 和平精英 上 战神 必备 技巧 的 游戏 视频,Video-Play
6,2019 年 古装 爱情 电视剧 小女 花不弃 的 花絮 播放 一下,Video-Play
7,找 一个 2004 年 的 推理剧 给 我 看 一会 呢,FilmTele-Play
8,自驾游 去 深圳 都 经过 那些 地方 啊,Travel-Query
9,给 我 转播 今天 的 女子双打 乒乓球 比赛 现场,Video-Play


In [4]:
datas.describe()

,0,1
count,12100,12100
unique,12073,12
top,中元节 是 几月 几号,FilmTele-Play
freq,3,1355


In [5]:
X = datas.iloc[:, 0]
y = datas.iloc[:, -1]

encoder = LabelEncoder()
y_encode = encoder.fit_transform(y)

print(encoder.classes_)
class2label = encoder.classes_

['Alarm-Update' 'Audio-Play' 'Calendar-Query' 'FilmTele-Play'
 'HomeAppliance-Control' 'Music-Play' 'Other' 'Radio-Listen'
 'TVProgram-Play' 'Travel-Query' 'Video-Play' 'Weather-Query']


In [6]:
X_train,X_test,y_train,y_test = train_test_split(X,y_encode,test_size=0.2,random_state=42)

In [7]:
vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(vectorizer.get_feature_names_out())
print(vectorizer.get_feature_names_out().shape)
print(X_train_tfidf.toarray())
print(X_train_tfidf.shape)

['00' '01' '05' ... '龚俊' '龚俊演' '龟兔']
(8776,)
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
(9680, 8776)


In [97]:
LR = LogisticRegression()
LR.fit(X_train_tfidf, y_train)
y_pred = LR.predict(X_test_tfidf)
print(y_pred)
print(LR.score(X_test_tfidf,y_test))

[ 9  4 11 ...  4  0  5]
0.8900826446280992


In [111]:
RFC = RandomForestClassifier()
RFC.fit(X_train_tfidf, y_train)
y_pred = RFC.predict(X_test_tfidf)
print(y_pred)
print(RFC.score(X_test_tfidf,y_test))

[ 9  4 11 ...  5  0  5]
0.846694214876033


In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader,Dataset

class Network_Classifier(nn.Module):
    def __init__(self,in_features,num_classes):
        super(Network_Classifier, self).__init__()
        self.fc_layers = nn.Sequential(
            nn.Linear(in_features,4096),
            nn.ReLU(),
            nn.Linear(4096,1024),
            nn.ReLU(),
            nn.Linear(1024,num_classes),
		)

    def forward(self,x):
        return self.fc_layers(x)

In [15]:
class MyDataset(Dataset):
    def __init__(self, data, target):

        if hasattr(data, "toarray"):
            data = data.toarray()

        data = np.array(data).astype('float32')
        target = np.array(target).astype('int64')

        self.data = torch.from_numpy(data)
        self.target = torch.from_numpy(target)

    def __len__(self):
        return self.data.shape[0]

    def __getitem__(self, idx):
        return self.data[idx], self.target[idx]

In [21]:
batch_size = 64
learning_rate = 0.01
epochs = 200
device = torch.device("mps" if torch.mps.is_available() else "cpu")

In [22]:
train_loader = DataLoader(MyDataset(X_train_tfidf,y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(MyDataset(X_test_tfidf,y_test), batch_size=int(batch_size/2), shuffle=False)

n,m = X_test_tfidf.shape
p = len(np.unique(y_train))

model = Network_Classifier(m,p).to(device)
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [23]:
model.train()
for epoch in range(epochs):
    for batch_idx, (data, target) in enumerate(train_loader):
        data,target = data.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        if batch_idx % 100 == 0:
            print(f'Train Epoch: {epoch} [{batch_idx * len(data)}/{len(train_loader.dataset)}] Loss: {loss.item():.6f}')


Train Epoch: 0 [0/9680] Loss: 2.485350
Train Epoch: 0 [6400/9680] Loss: 2.466613
Train Epoch: 1 [0/9680] Loss: 2.464716
Train Epoch: 1 [6400/9680] Loss: 2.442105
Train Epoch: 2 [0/9680] Loss: 2.432231
Train Epoch: 2 [6400/9680] Loss: 2.439576
Train Epoch: 3 [0/9680] Loss: 2.415153
Train Epoch: 3 [6400/9680] Loss: 2.385559
Train Epoch: 4 [0/9680] Loss: 2.421940
Train Epoch: 4 [6400/9680] Loss: 2.452553
Train Epoch: 5 [0/9680] Loss: 2.412160
Train Epoch: 5 [6400/9680] Loss: 2.353903
Train Epoch: 6 [0/9680] Loss: 2.362724
Train Epoch: 6 [6400/9680] Loss: 2.379432
Train Epoch: 7 [0/9680] Loss: 2.361529
Train Epoch: 7 [6400/9680] Loss: 2.367631
Train Epoch: 8 [0/9680] Loss: 2.371280
Train Epoch: 8 [6400/9680] Loss: 2.355059
Train Epoch: 9 [0/9680] Loss: 2.331814
Train Epoch: 9 [6400/9680] Loss: 2.359132
Train Epoch: 10 [0/9680] Loss: 2.358633
Train Epoch: 10 [6400/9680] Loss: 2.347937
Train Epoch: 11 [0/9680] Loss: 2.371644
Train Epoch: 11 [6400/9680] Loss: 2.347020
Train Epoch: 12 [0/9680]

In [24]:
model.eval()
test_loss = 0
correct = 0
with torch.no_grad():
    for data, target in test_loader:
        data, target = data.to(device), target.to(device)
        output = model(data)
        test_loss += criterion(output, target).item()
        pred = output.argmax(dim=1, keepdim=True)
        # output.sort(dim=1, descending=True)

        correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f'\nTest set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)\n')


Test set: Average loss: 0.0151, Accuracy: 2093/2420 (86.49%)



Connected to: <socket.socket fd=84, family=2, type=1, proto=0, laddr=('127.0.0.1', 55231), raddr=('127.0.0.1', 55126)>.


In [119]:

def predict(text,k,model):
    import jieba
    text_cut = [" ".join(jieba.lcut(i)) for i in text]
    # print("分割结果：",text_cut)
    text_cut_vector = vectorizer.transform(text_cut)
    text_pre = model.predict(text_cut_vector)
    texr_pre_proba = model.predict_proba(text_cut_vector)
    text_pre_k = np.argsort(-texr_pre_proba,axis=1)[:,:k]
    texr_pre_pro = F.softmax(texr_pre_proba)[:,k]
    print(texr_pre_pro)
    print(f"{model}概率最大的{k}个类型：",class2label[text_pre_k])
    print(f"{model}概率最大的类型：",class2label[text_pre])


In [124]:
text = ["我听王力宏",
        "我听陶喆",
        "我看饺子的哪吒2",
        "我看周星驰的功夫"]
k = 3

predict(text,k,LR)
predict(text,k,RFC)

LogisticRegression()概率最大的3个类型： [['Music-Play' 'Video-Play' 'FilmTele-Play']
 ['FilmTele-Play' 'Music-Play' 'Radio-Listen']
 ['FilmTele-Play' 'Video-Play' 'Music-Play']
 ['FilmTele-Play' 'Music-Play' 'Video-Play']]
LogisticRegression()概率最大的类型： ['Music-Play' 'FilmTele-Play' 'FilmTele-Play' 'FilmTele-Play']
RandomForestClassifier()概率最大的3个类型： [['Music-Play' 'FilmTele-Play' 'Radio-Listen']
 ['Music-Play' 'FilmTele-Play' 'Radio-Listen']
 ['Music-Play' 'FilmTele-Play' 'Radio-Listen']
 ['FilmTele-Play' 'Music-Play' 'HomeAppliance-Control']]
RandomForestClassifier()概率最大的类型： ['Music-Play' 'Music-Play' 'Music-Play' 'FilmTele-Play']
